# Waypoint — End-to-End Demo

This notebook walks through the full Waypoint workflow using real market data fetched from **yfinance** and **FRED**.

Sections:
1. Fetch real asset data (yfinance + FRED)
2. Portfolio construction
3. Expected Return (HistoricalMean)
4. Risk (SampleCovariance)
5. Efficient Frontier (Optimizer)
6. Wealth Simulation (MonteCarlo, inflation-adjusted withdrawals)

## 1. Fetch Real Data

We fetch 10 years of daily returns (2015–2024) for three assets via **yfinance**, and CPI from **FRED** to estimate long-run inflation.

| Asset | Symbol | Vendor |
|---|---|---|
| US Large Cap Equities | ^SPX | yfinance |
| Intl Developed Equities | EFA | yfinance |
| US Aggregate Bonds | AGG | yfinance |
| CPI YoY | CPIAUCSL | FRED |

**Prerequisites:**
- yfinance: `uv sync --extra yfinance`
- FRED: `uv sync --extra fred` + set `FRED_API_KEY` env var (free key at fred.stlouisfed.org)

Data is cached locally in parquet after the first fetch — re-running this notebook is instant.

In [1]:
import numpy as np
import polars as pl

import waypoint as wp

START = "2015-01-01"
END   = "2024-12-31"
FREQ  = wp.Frequency.DAILY

# --- yfinance ---
equities = wp.fetch(wp.catalog.US_LARGE_CAP,   start=START, end=END)
intl     = wp.fetch(wp.catalog.INTL_DEVELOPED, start=START, end=END)
bonds    = wp.fetch(wp.catalog.US_AGG_BONDS,   start=START, end=END)

for asset in [equities, intl, bonds]:
    print(f"{asset.name:30s}  {len(asset.returns):>5} days  "
          f"{asset.returns['date'].min()} → {asset.returns['date'].max()}")

# --- FRED: CPI for inflation estimate ---
try:
    cpi = wp.fetch(wp.catalog.CPI_YOY, start=START, end=END)
    monthly_cpi = cpi.returns["returns"].to_numpy()
    annual_inflation = float((1 + monthly_cpi).prod() ** (12 / len(monthly_cpi)) - 1)
    print(f"\n{cpi.name:30s}  {len(cpi.returns):>5} months")
    print(f"Estimated long-run annual inflation: {annual_inflation:.2%}")
except OSError as e:
    annual_inflation = 0.03
    print(f"\nFRED unavailable ({e}) — using default inflation {annual_inflation:.0%}")

# --- FRED: 10Y yield for risk-free rate ---
try:
    rf_indicator = wp.fetch(wp.catalog.US_10Y_YIELD, start=START, end=END)
    risk_free_rate = float(rf_indicator.values["value"].tail(1).item()) / 100
    print(f"Risk-free rate (latest DGS10): {risk_free_rate:.2%}")
except OSError:
    risk_free_rate = 0.04
    print(f"FRED unavailable — using default risk-free rate {risk_free_rate:.0%}")

$^SPX: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-01-02)
$EFA: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-01-02)
$AGG: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-01-02)


US Large Cap Equities            2515 days  2015-01-05 → 2024-12-31
Intl Developed Equities          2515 days  2015-01-05 → 2024-12-31
US Aggregate Bonds               2515 days  2015-01-05 → 2024-12-31

CPI YoY                           119 months
Estimated long-run annual inflation: 3.10%
Risk-free rate (latest DGS10): 4.58%


## 2. Create Portfolio — 60/20/20 Initial Weights

Construct a portfolio with 60% US equities, 20% international equities, 20% bonds.

In [2]:
portfolio = wp.Portfolio(
    slots={
        "US Equities": equities,
        "Intl Equities": intl,
        "US Bonds": bonds,
    },
    weights={
        "US Equities": 0.60,
        "Intl Equities": 0.20,
        "US Bonds": 0.20,
    },
    name="60/20/20 Global",
)

print("Portfolio weights:")
for name, weight in portfolio.weights.items():
    print(f"  {name}: {weight:.1%}")

wide = portfolio.get_returns()
print(f"\nAligned returns: {len(wide)} days  "
      f"{wide['date'].min()} → {wide['date'].max()}")
print(wide.head())

Portfolio weights:
  US Equities: 60.0%
  Intl Equities: 20.0%
  US Bonds: 20.0%

Aligned returns: 2515 days  2015-01-05 → 2024-12-31
shape: (5, 4)
┌────────────┬─────────────┬───────────────┬───────────┐
│ date       ┆ US Equities ┆ Intl Equities ┆ US Bonds  │
│ ---        ┆ ---         ┆ ---           ┆ ---       │
│ date       ┆ f64         ┆ f64           ┆ f64       │
╞════════════╪═════════════╪═══════════════╪═══════════╡
│ 2015-01-05 ┆ -0.018278   ┆ -0.023605     ┆ 0.002173  │
│ 2015-01-06 ┆ -0.008893   ┆ -0.011327     ┆ 0.002529  │
│ 2015-01-07 ┆ 0.01163     ┆ 0.011115      ┆ -0.00018  │
│ 2015-01-08 ┆ 0.017888    ┆ 0.013529      ┆ -0.001532 │
│ 2015-01-09 ┆ -0.008404   ┆ -0.004839     ┆ 0.002438  │
└────────────┴─────────────┴───────────────┴───────────┘


## 3. Expected Return — HistoricalMean

Estimate annualised expected returns using the arithmetic historical mean.

In [3]:
er_model = wp.ExpectedReturn(method=wp.HistoricalMean())
er_result = er_model.compute(portfolio, start=None, end=None, frequency=FREQ)

print(f"Method: {er_result.method_name}")
print("Per-asset annualised expected returns:")
for name, ret in er_result.per_asset.items():
    print(f"  {name}: {ret:.2%}")
print(f"Portfolio expected return: {er_result.portfolio:.2%}")

Method: HistoricalMean
Per-asset annualised expected returns:
  US Equities: 12.12%
  Intl Equities: 6.63%
  US Bonds: 1.42%
Portfolio expected return: 8.88%


## 4. Risk — SampleCovariance

Estimate the annualised covariance matrix and per-asset volatilities.

In [4]:
risk_model = wp.Risk(method=wp.SampleCovariance())
risk_result = risk_model.compute(portfolio, start=None, end=None, frequency=FREQ)

print(f"Method: {risk_result.method_name}")
print()
print("Annualised covariance matrix:")
print(risk_result.covariance)
print()
print("Per-asset annualised volatility:")
for name, vol in risk_result.volatilities.items():
    print(f"  {name}: {vol:.2%}")
print(f"Portfolio volatility: {risk_result.portfolio_volatility:.2%}")

Method: SampleCovariance

Annualised covariance matrix:
shape: (3, 3)
┌─────────────┬───────────────┬──────────┐
│ US Equities ┆ Intl Equities ┆ US Bonds │
│ ---         ┆ ---           ┆ ---      │
│ f64         ┆ f64           ┆ f64      │
╞═════════════╪═══════════════╪══════════╡
│ 0.031783    ┆ 0.02629       ┆ 0.000852 │
│ 0.02629     ┆ 0.029863      ┆ 0.001221 │
│ 0.000852    ┆ 0.001221      ┆ 0.00282  │
└─────────────┴───────────────┴──────────┘

Per-asset annualised volatility:
  US Equities: 17.83%
  Intl Equities: 17.28%
  US Bonds: 5.31%
Portfolio volatility: 13.91%


## 5. Efficient Frontier

Build the mean-variance efficient frontier using the `Optimizer`.
Constraints: long-only, weights sum to 1.

In [5]:
optimizer = wp.Optimizer(
    return_model=wp.ExpectedReturn(method=wp.HistoricalMean()),
    risk_model=wp.Risk(method=wp.SampleCovariance()),
    constraints=[wp.LongOnly(), wp.SumToOne()],
)

frontier = optimizer.efficient_frontier(
    portfolio, start=None, end=None,
    frequency=FREQ, n_points=30,
)

print(f"Frontier points computed: {len(frontier.weights)}")
print("\nFirst 5 frontier points (risk ascending):")
summary = pl.DataFrame({
    "risk": frontier.risks,
    "expected_return": frontier.expected_returns,
})
print(summary.head())

print()
# Use risk-free rate fetched from FRED above
sharpe_weights = frontier.optimal_sharpe(risk_free_rate=risk_free_rate)
print(f"Max Sharpe weights (rf = {risk_free_rate:.2%}):")
for name, w in sharpe_weights.items():
    print(f"  {name}: {w:.1%}")

Frontier points computed: 30

First 5 frontier points (risk ascending):
shape: (5, 2)
┌──────────┬─────────────────┐
│ risk     ┆ expected_return │
│ ---      ┆ ---             │
│ f64      ┆ f64             │
╞══════════╪═════════════════╡
│ 0.051984 ┆ 0.020648        │
│ 0.051984 ┆ 0.020648        │
│ 0.05201  ┆ 0.021625        │
│ 0.052582 ┆ 0.025312        │
│ 0.053879 ┆ 0.029           │
└──────────┴─────────────────┘

Max Sharpe weights (rf = 4.58%):
  US Equities: 100.0%
  Intl Equities: 0.0%
  US Bonds: -0.0%


In [6]:
# Plot the efficient frontier
fig = frontier.plot()
fig.show()

## 6. Wealth Simulation — Monte Carlo

Simulate 20-year wealth paths starting from $1,000,000 with an annual $100,000 inflation-adjusted contribution.
The inflation rate comes from FRED CPI.  We show both **nominal** and **real** (inflation-adjusted) fan charts with dates on the x-axis.

In [7]:
from datetime import date

contribution = wp.PeriodicCashflow(
    amount=100_000.0,
    frequency=wp.Frequency.ANNUAL,
    mode=wp.CashflowMode.DOLLAR,
    inflation_rate=annual_inflation,  # from FRED CPI
)

wealth_sim = wp.WealthSimulation(
    method=wp.MonteCarlo(seed=42),
    cashflows=[contribution],
    horizon_years=20,
    initial_wealth=1_000_000.0,
    n_simulations=1000,
    inflation_rate=annual_inflation,  # used for real deflation
)

# Nominal — values in future dollars
sim_nominal = wealth_sim.compute(
    portfolio, start=None, end=None, frequency=FREQ,
    start_date=END,   # projection starts at end of historical window
    real=False,
)

# Real — values in today's purchasing power
sim_real = wealth_sim.compute(
    portfolio, start=None, end=None, frequency=FREQ,
    start_date=END,
    real=True,
)

print(f"Simulation paths shape: {sim_nominal.paths.shape}")
print(f"X-axis: {sim_nominal.percentile_df['date'][0]}  →  {sim_nominal.percentile_df['date'][-1]}")
print()

for label, result in [("Nominal", sim_nominal), ("Real", sim_real)]:
    s = result.summary()
    print(f"{label} terminal wealth (20 years):")
    print(f"  5th percentile:  ${s['p5_terminal']:>14,.0f}")
    print(f"  Median:          ${s['median_terminal']:>14,.0f}")
    print(f"  95th percentile: ${s['p95_terminal']:>14,.0f}")
    print()

Simulation paths shape: (1000, 5041)
X-axis: 2024-12-31  →  2044-12-31

Nominal terminal wealth (20 years):
  5th percentile:  $     5,357,005
  Median:          $    11,363,230
  95th percentile: $    24,876,164

Real terminal wealth (20 years):
  5th percentile:  $     2,911,680
  Median:          $     6,176,230
  95th percentile: $    13,520,885



In [8]:
# Nominal fan chart — future dollars, x-axis shows calendar dates
sim_nominal.plot().show()

In [9]:
# Real fan chart — today's purchasing power, same date x-axis
sim_real.plot().show()